# `json2vec` Hello World

This notebook trains the smallest useful JSON2Vec model: two numeric Iris measurements predict the Iris species. The point is not accuracy. The point is to show the complete loop from records, to schema, to training, to prediction and embeddings. This example is intentionally flat; the next tutorials show why arrays matter.

Start with the normal training dependencies plus the bundled Iris JSONL buffer. The examples remove notebook logging noise so the rendered docs stay focused on model behavior.


In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()


Load a tiny balanced slice of Iris rows. The schema field names match the DataFrame columns, so JSON2Vec can infer the request queries.


In [2]:
records = pl.read_ndjson("docs/data/iris.jsonl").head(36)

records.head()


sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""setosa"""
7.0,3.2,4.7,1.4,"""versicolor"""
6.3,3.3,6.0,2.5,"""virginica"""
4.9,3.0,1.4,0.2,"""setosa"""
6.4,3.2,4.5,1.5,"""versicolor"""


The schema declares exactly what the model should read. `Number` fields become numeric tensorfields, and the `Category` field is a supervised target because `target=True` hides it from the input and asks the model to decode it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("sepal_length"),
    j2v.Number("petal_length"),
    j2v.Category("species", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)


Train for one deliberately small epoch. The tutorials keep batch and epoch counts hardcoded so the example remains quick to run in documentation builds.

In [4]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

Prediction uses the same nested batch shape as training: each outer item is one observation, and each observation contains one record.

In [5]:
batch = [[record] for record in records.to_dicts()[:3]]

In [6]:
pprint(model.predict(batch))

{
│   'record/species': {
│   │   'state': {
│   │   │   'valued': [0.3269572854042053, 0.31602126359939575, 0.3170427083969116],
│   │   │   'null': [0.16281504929065704, 0.17097513377666473, 0.16776347160339355],
│   │   │   'padded': [0.1509781926870346, 0.1550551950931549, 0.1549673080444336],
│   │   │   'masked': [0.14830122888088226, 0.13995708525180817, 0.14305052161216736],
│   │   │   'other': [0.21094827353954315, 0.21799123287200928, 0.21717606484889984]
│   │   },
│   │   'content': {
│   │   │   'value': ['virginica', 'virginica', 'virginica'],
│   │   │   'probability': [0.4025634527206421, 0.3872278034687042, 0.3966466188430786],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'virginica', 'probability': 0.4025634527206421},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.3342014253139496}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'virginica', 'probability': 0.3872278034687042},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.35328879952430725}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'virginica', 'probability': 0.3966466188430786},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.3406282961368561}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

Embeddings are opt-in. Passing `embed=True` when constructing the model exposes a root `record` vector for each input observation without changing the schema fields themselves.


In [7]:
pprint(model.embed(batch))


{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.0248271431773901,
│   │   │   │   0.07523186504840851,
│   │   │   │   0.2673119008541107,
│   │   │   │   -0.1830676943063736,
│   │   │   │   -0.11101137101650238,
│   │   │   │   0.3630831241607666,
│   │   │   │   0.11396843194961548,
│   │   │   │   0.20713941752910614,
│   │   │   │   -0.008094566874206066,
│   │   │   │   0.12465698271989822,
│   │   │   │   0.17863376438617706,
│   │   │   │   -0.56038898229599,
│   │   │   │   -0.01660725474357605,
│   │   │   │   0.2153267115354538,
│   │   │   │   -0.12275832146406174,
│   │   │   │   -0.5152126550674438
│   │   │   ],
│   │   │   [
│   │   │   │   -0.016668327152729034,
│   │   │   │   0.017790790647268295,
│   │   │   │   0.26329946517944336,
│   │   │   │   -0.13249360024929047,
│   │   │   │   -0.16748002171516418,
│   │   │   │   0.3718135952949524,
│   │   │   │   0.12675079703330994,
│   │   │   │   0.2002861350774765,
│   │   │   │   -0.014063682407140732,
│   │   │   │   0.18227829039096832,
│   │   │   │   0.18169550597667694,
│   │   │   │   -0.5577038526535034,
│   │   │   │   -0.0334937684237957,
│   │   │   │   0.23102760314941406,
│   │   │   │   -0.16825224459171295,
│   │   │   │   -0.47934842109680176
│   │   │   ],
│   │   │   [
│   │   │   │   0.009541342034935951,
│   │   │   │   0.03743285685777664,
│   │   │   │   0.2752278447151184,
│   │   │   │   -0.12412656098604202,
│   │   │   │   -0.15188677608966827,
│   │   │   │   0.3446391820907593,
│   │   │   │   0.1341540664434433,
│   │   │   │   0.21594808995723724,
│   │   │   │   -0.0003368014586158097,
│   │   │   │   0.13743360340595245,
│   │   │   │   0.16594089567661285,
│   │   │   │   -0.5755592584609985,
│   │   │   │   0.00018437416292726994,
│   │   │   │   0.1922471523284912,
│   │   │   │   -0.14127382636070251,
│   │   │   │   -0.5160068273544312
│   │   │   ]
│   │   ]
│   }
}

The plot is the quickest way to verify what was built: array nodes, tensorfield nodes, targets, embeddings, and parameter counts all appear in the same tree.

In [8]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  18,153                                │
│  Schema map                                                            arrays  1                                     │
│                                                                        fields  3                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               